# Enterprise RAG — Hands-On, Part 5 of 11: Why hybrid search, demonstrated

*Split from `02-hands-on.ipynb` for focused reading — same content, one phase at a time. The
"Setup" cell below re-derives whatever state earlier parts would have produced, so this notebook
runs standalone; you do not need to run the other parts first.*

**Prerequisites:** `OPENAI_API_KEY` in the repo-root `.env`, and `python scripts/ingest.py` already
run (the setup cell below will build the index for you if it is missing).

**Series:** [1. The corpus and its permissions](part01-corpus-and-permissions.ipynb) · [2. The policy engine](part02-policy-engine.ipynb) · [3. Compiling the policy into a database filter](part03-compiling-policy-to-filter.ipynb) · [4. Chunking and ingestion](part04-chunking-and-ingestion.ipynb) · [5. Why hybrid search, demonstrated](part05-hybrid-search.ipynb) · [6. Query transformation](part06-query-transformation.ipynb) · [7. Reranking](part07-reranking.ipynb) · [8. The full graph](part08-full-graph.ipynb) · [9. Attacking it](part09-attacking-it.ipynb) · [10. Evaluation](part10-evaluation.ipynb) · [11. Observability, and what to take away](part11-observability-and-takeaways.ipynb)

---


In [ ]:
import sys, json, textwrap
from pathlib import Path

# The package lives in src/ - add it to the path so this notebook runs from anywhere.
ROOT = Path.cwd()
while not (ROOT / "src" / "enterprise_rag").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from enterprise_rag.config import SETTINGS

print("project root :", ROOT)
print("corpus       :", SETTINGS.corpus_dir.relative_to(ROOT))
print("api key      :", "found" if SETTINGS.has_api_key else "MISSING - check .env")
print("embed model  :", SETTINGS.embedding_model)
print("chat model   :", SETTINGS.fast_model)

### Setup — recap of state from earlier parts


In [ ]:
from enterprise_rag.identity import get_principal
from enterprise_rag.authz.policy import compile_prefilter
from enterprise_rag.ingest.store import fetch_all_allowed

# Build the index if it is not already there (same check as Part 4).
from enterprise_rag.ingest.store import collection_stats
try:
    stats = collection_stats("meridian")
    assert stats["chunks"] > 0
except Exception:
    from enterprise_rag.ingest.pipeline import ingest
    print("building index...")
    print(ingest().render())

---
# Part 5 - Why hybrid search, demonstrated

The claim from the theory doc: **dense search finds things that *mean* the same; lexical search finds
things that *say* the same.** Enterprise corpora need both.

Let's prove it rather than assert it.

In [ ]:
from enterprise_rag.llm.client import LLMClient
from enterprise_rag.ingest import store
from enterprise_rag.retrieval.lexical import BM25Index

llm = LLMClient()
lena = get_principal("u_lena_t1")
where = compile_prefilter(lena)
pool = fetch_all_allowed(lena.tenant_id, where)
bm25 = BM25Index(pool)

def compare(question, k=4):
    vec = llm.embed([question])[0]
    dense = store.dense_search(lena.tenant_id, vec, where, k)
    lex = bm25.search(question, k)
    print(f'Q: "{question}"')
    print(f"  {'DENSE (meaning)':<34}{'BM25 (exact words)'}")
    for i in range(k):
        d = f"{dense[i].chunk.chunk_id} ({dense[i].score:.3f})" if i < len(dense) else "-"
        l = f"{lex[i].chunk.chunk_id} ({lex[i].score:.1f})" if i < len(lex) else "-"
        print(f"  {d:<34}{l}")
    print()

compare("What does MRD-4290 mean?")                              # rare identifier
compare("telemetry disappears before it is saved permanently")   # pure paraphrase

ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given
ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given
ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given


Q: "What does MRD-4290 mean?"
  DENSE (meaning)                   BM25 (exact words)
  HC-002#3 (0.454)                  HC-002#3 (2.4)
  HC-002#1 (0.427)                  HC-003#1 (2.2)
  HC-002#0 (0.378)                  HC-002#1 (2.0)
  TK-4471#2 (0.357)                 HC-002#2 (1.9)



ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


Q: "telemetry disappears before it is saved permanently"
  DENSE (meaning)                   BM25 (exact words)
  HC-002#1 (0.339)                  HC-001#0 (4.9)
  TK-4502#0 (0.322)                 HC-004#1 (3.5)
  TK-4502#2 (0.320)                 HC-001#2 (2.8)
  TK-4502#1 (0.313)                 HC-002#0 (1.8)



Look at what each one actually got (your scores may shift slightly between runs):

**`MRD-4290`** - both retrievers surface `HC-002#3`, which is the *"Related codes"* footnote that
merely mentions `MRD-4290`. But `MRD-4290` genuinely belongs to `HC-003` (Rate Limits), and only
**BM25 surfaces `HC-003#1`**. Dense retrieval fills its whole top-3 with `HC-002` - the doc about
`MRD-5031`. That is the failure mode in action: `MRD-5031`, `MRD-5030` and `MRD-4290` *look* alike, so
they embed to nearly the same place, and dense search cannot tell them apart.

**"telemetry disappears before it is saved permanently"** - not one content word is shared with the
answer. Dense retrieval still finds `HC-002#1`, the passage stating that data receiving `MRD-5031`
*"has not been durably stored"*. BM25 is pulled to `HC-001#0` (Getting Started) by common words like
*"saved"* and *"data"* - topically plausible, factually useless.

So: **dense understands meaning but blurs identifiers; lexical nails identifiers but is fooled by
common words.** Neither wins alone. Now fuse them.

In [ ]:
from enterprise_rag.retrieval.fusion import reciprocal_rank_fusion

q = "What does MRD-4290 mean?"
vec = llm.embed([q])[0]
dense = store.dense_search(lena.tenant_id, vec, where, 8)
lex = bm25.search(q, 8)
fused = reciprocal_rank_fusion([dense, lex], top_n=5)

print(f"{'chunk':<16}{'RRF score':<12}{'found by'}")
print("-" * 46)
for sc in fused:
    print(f"{sc.chunk.chunk_id:<16}{sc.fused_score:<12.5f}{'+'.join(sc.retrieved_by)}")
print("\nChunks found by BOTH retrievers rise to the top - that agreement")
print("across independent methods is the signal RRF is designed to reward.")

ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


chunk           RRF score   found by
----------------------------------------------
HC-002#3        0.03279     dense+bm25
HC-002#1        0.03200     dense+bm25
HC-003#1        0.03128     dense+bm25
HC-002#2        0.03055     dense+bm25
HC-002#0        0.01587     dense

Chunks found by BOTH retrievers rise to the top - that agreement
across independent methods is the signal RRF is designed to reward.


---

**◀ Previous:** [4. Chunking and ingestion](part04-chunking-and-ingestion.ipynb)

**Next ▶:** [6. Query transformation](part06-query-transformation.ipynb)
